# A full training

# Install

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [56]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

# Check GPU is connected

In [57]:
import torch

In [58]:
!nvidia-smi

Thu Jun  6 15:34:39 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 551.61                 Driver Version: 551.61         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA T1000 8GB             WDDM  |   00000000:65:00.0 Off |                  N/A |
| 33%   34C    P8             N/A /   50W |    3391MiB /   8192MiB |      8%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [59]:

if torch.cuda.is_available():
    print("CUDA is available! 🎉")
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("CUDA is not available. 🙁")

CUDA is available! 🎉
Number of CUDA devices: 1


In [60]:
import torch
import sys
print('__Python VERSION:', sys.version)
print('__pyTorch VERSION:', torch.__version__)
print('__CUDA VERSION')
from subprocess import call
# call(["nvcc", "--version"]) does not work
! nvcc --version
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('__Number CUDA Devices:', torch.cuda.device_count())
print('__Devices')
call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])
print('Active CUDA Device: GPU', torch.cuda.current_device())
print ('Available devices ', torch.cuda.device_count())
print ('Current cuda device ', torch.cuda.current_device())

__Python VERSION: 3.11.7 | packaged by Anaconda, Inc. | (main, Dec 15 2023, 18:05:47) [MSC v.1916 64 bit (AMD64)]
__pyTorch VERSION: 2.3.1
__CUDA VERSION
nvcc: NVIDIA (R) Cuda compiler driver__CUDNN VERSION: 8907
__Number CUDA Devices: 1
__Devices

Copyright (c) 2005-2021 NVIDIA Corporation
Built on Fri_Dec_17_18:28:54_Pacific_Standard_Time_2021
Cuda compilation tools, release 11.6, V11.6.55
Build cuda_11.6.r11.6/compiler.30794723_0
Active CUDA Device: GPU 0
Available devices  1
Current cuda device  0


In [61]:
# activate GPU device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cuda:0


# Load dataset

In [62]:
from datasets import load_dataset

raw_datasets = load_dataset("glue","mrpc")
# raw_datasets = load_dataset("humarin/chatgpt-paraphrases")



In [63]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [64]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [65]:
raw_train_dataset.features

{'sentence1': Value(dtype='string', id=None),
 'sentence2': Value(dtype='string', id=None),
 'label': ClassLabel(names=['not_equivalent', 'equivalent'], id=None),
 'idx': Value(dtype='int32', id=None)}

# Fine tuning a model

In [66]:
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModel
import torch
import torch.nn.functional as F

In [67]:

checkpoint = 'sentence-transformers/all-mpnet-base-v2'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

model.to(device) # Move model to the device



MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [68]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

# prepare for training

In [69]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'attention_mask']

In [70]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [71]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[51, 60, 48, 68, 60, 51, 63, 33]

In [72]:
["attention_mask", "input_ids", "labels", "token_type_ids"]

['attention_mask', 'input_ids', 'labels', 'token_type_ids']

In [73]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

In [74]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 71]),
 'attention_mask': torch.Size([8, 71])}

In [75]:
# Move the batch of data to the same device
for key in batch.keys():
    batch[key] = batch[key].to(device)

# train the model

## train with trainer

In [76]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

In [77]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [78]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [79]:
trainer.train()

  0%|          | 0/1377 [00:00<?, ?it/s]

{'loss': 0.6199, 'grad_norm': 0.3589082956314087, 'learning_rate': 3.184458968772695e-05, 'epoch': 1.09}
{'loss': 0.6114, 'grad_norm': 0.7037774920463562, 'learning_rate': 1.3689179375453886e-05, 'epoch': 2.18}
{'train_runtime': 408.7233, 'train_samples_per_second': 26.923, 'train_steps_per_second': 3.369, 'train_loss': 0.5933923658981822, 'epoch': 3.0}


TrainOutput(global_step=1377, training_loss=0.5933923658981822, metrics={'train_runtime': 408.7233, 'train_samples_per_second': 26.923, 'train_steps_per_second': 3.369, 'total_flos': 410769801850080.0, 'train_loss': 0.5933923658981822, 'epoch': 3.0})

In [80]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

  0%|          | 0/51 [00:00<?, ?it/s]

(408, 2) (408,)


In [81]:
validation_dataset = tokenized_datasets["validation"]
validation_dataset[0]

{'labels': tensor(1),
 'input_ids': tensor([    0,  2006,  2060,  2000,  9444,  2125,  7907,  2067, 11349,  2453,
          2991,  1009,  1060,  4910,  2000,  2198,  1009,  1059,  2150,  1015,
          2748,  3934,  5660,  1016,     2,     2,  1004,  2000,  9444,  2125,
          7907,  2067, 11349,  2453,  2519,  2029,  4910,  2260,  2150,  1015,
          2748,  3934,  5660,  1016,     2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}

In [82]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)
preds

array([1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,

## Print validation set

In [83]:
raw_validation_dataset = raw_datasets["validation"]
raw_validation_dataset[0]

{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .",
 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .',
 'label': 1,
 'idx': 9}

In [84]:
raw_validation_dataset


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})

In [85]:
predictions.predictions

array([[-0.9654739 ,  0.8255031 ],
       [-0.23548844,  0.25240254],
       [-0.3195506 ,  0.31343788],
       [-0.9652667 ,  0.82224584],
       [ 0.09514002, -0.04194226],
       [-0.965623  ,  0.8236726 ],
       [-0.9612434 ,  0.8144436 ],
       [-0.9656168 ,  0.82429117],
       [-0.96594375,  0.82449436],
       [-0.9656021 ,  0.82528055],
       [-0.9664625 ,  0.82344306],
       [-0.9648395 ,  0.82194257],
       [ 0.62844974, -0.57652587],
       [-0.8162479 ,  0.67419475],
       [-0.96548754,  0.8251813 ],
       [-0.9658094 ,  0.8251584 ],
       [-0.96508074,  0.8256155 ],
       [ 0.5745433 , -0.5310082 ],
       [-0.96563363,  0.82544345],
       [-0.23412307,  0.25149953],
       [ 0.52212703, -0.49608713],
       [-0.49480408,  0.42912075],
       [-0.23541355,  0.25213888],
       [-0.9655761 ,  0.8251026 ],
       [-0.9657095 ,  0.82504946],
       [-0.06027763,  0.10776708],
       [ 0.3079047 , -0.23149922],
       [-0.96531355,  0.82548183],
       [-0.9635512 ,

# evaluate

In [30]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.875, 'f1': 0.911917098445596}

## Inference

Let's test the model on a new sentence:

In [31]:
sentence1 = "Im happy to try to train a model that works for semantic similarity"
sentence2 = "trainig a model that workds for semantic similarity makes me happy"
sentence3  = "I am climbing and looking for a climbing partner"

In [32]:

encoding = tokenizer(sentence2, sentence3, return_tensors="pt")
encoding = {k: v.to(trainer.model.device) for k,v in encoding.items()}

outputs = trainer.model(**encoding)
print(outputs)



SequenceClassifierOutput(loss=None, logits=tensor([[ 1.6926, -1.7196]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


The logits that come out of the model are of shape (batch_size, num_labels). As we are only forwarding a single sentence through the model, the `batch_size` equals 1. The logits is a tensor that contains the (unnormalized) scores for every individual label.

In [33]:
logits = outputs.logits

predictions = torch.argmax(logits, dim=-1)

# logits.shape
print(predictions)

tensor([0], device='cuda:0')


To turn them into actual predicted labels, we first apply a sigmoid function independently to every score, such that every score is turned into a number between 0 and 1, that can be interpreted as a "probability" for how certain the model is that a given class belongs to the input text.

Next, we use a threshold (typically, 0.5) to turn every probability into either a 1 (which means, we predict the label for the given example) or a 0 (which means, we don't predict the label for the given example).

In [34]:
# # apply sigmoid + threshold
# sigmoid = torch.nn.Sigmoid()
# probs = sigmoid(logits.squeeze().cpu())
# predictions = np.zeros(probs.shape)
# predictions[np.where(probs >= 0.5)] = 1
# # turn predicted id's into actual label names
# predicted_labels = [id2label[idx] for idx, label in enumerate(predictions) if label == 1.0]
# print(predicted_labels)

In [50]:
trainer.save_model("./my_model")

In [55]:
model.save_to_hub("my_new_model")

AttributeError: 'MPNetModel' object has no attribute 'save_to_hub'

# Use the pretrained model for PHRAME

In [ ]:
sentence1 = "Im happy to try to train a model that works for semantic similarity"
sentence2 = "trainig a model that workds for semantic similarity makes me happy"
sentence3  = "I am climbing and looking for a climbing partner"


# load pretrained model

In [54]:
model = AutoModel.from_pretrained("./my_model")

Some weights of MPNetModel were not initialized from the model checkpoint at ./my_model and are newly initialized: ['mpnet.pooler.dense.bias', 'mpnet.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
!pip install sentence_transformers

  Using cached sentence_transformers-3.0.0-py3-none-any.whl.metadata (10 kB)
Using cached sentence_transformers-3.0.0-py3-none-any.whl (224 kB)


In [46]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers import SentenceTransformer, models, InputExample, losses


In [47]:
model = SentenceTransformer("./my_model")


No sentence-transformers model found with name ./my_model. Creating a new one with mean pooling.
Some weights of MPNetModel were not initialized from the model checkpoint at ./my_model and are newly initialized: ['mpnet.pooler.dense.bias', 'mpnet.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
from huggingface_hub import notebook_login

In [38]:
notebook_login("./my_model")

In [39]:
trainer.push_to_hub("your-username/model-name")

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-666208e0-05cfe6b83620f7f6559b010e;47354274-d5d9-4ad2-9955-682a34f255fc)

Invalid username or password.

# (IGNORE) training without trainer

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [ ]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)